In [1]:
import mcelect
from mcelect.config import *
import awkward as ak
import numpy as np
import hvplot.pandas  # noqa
import pandas as pd
import panel as pn

In [2]:
mw_config = Config(
    voters=600,
    candidates=12,
    mode="multi_winner",
    committee_size=5,
    considerations=[
        Electorate(
            dimensions=2,
            distance_function=QGaussian2(1.0),
            factions=[
                Faction(
                    popularity=1.0, voter_center=[-2.0, 1.0], voter_spread=1.0, candidate_spread=0.5
                ),
                Faction(
                    popularity=0.95, voter_center=[2.0, 1.0], voter_spread=1.0, candidate_spread=0.5
                ),
                Faction(
                    popularity=0.5,
                    voter_center=[-2.0, -1.0],
                    voter_spread=1.0,
                    candidate_spread=0.5,
                ),
                # Faction(popularity=1.0, voter_center=[2.0, -1.0], voter_spread=1.0, candidate_spread= 0.5 ),
            ],
        )
    ],
    committee_methods=[
        PluralityTopN(),
        RRV(strat="Honest", ranks=10, k=0.5),
    ],
)

In [3]:
table = mcelect.simulate(mw_config, 10000)
results = ak.from_arrow(table)

In [4]:
results.methods.rrv_10_h.winners

<Array [[0, 7, 2, 3, 4], [...], ..., [0, 7, 1, 4, 8]] type='10000 * 5 * uint32'>

In [8]:
rrv_winners = ak.from_regular(results.methods.rrv_10_h.winners, axis=1)
pltn_winners = ak.from_regular(results.methods.pltn.winners, axis=1)

In [9]:
rrv_factions = ak.flatten(results.electorate.faction[rrv_winners])
pltn_factions = ak.flatten(results.electorate.faction[pltn_winners])
rrv_positions = ak.flatten(results.electorate.positions[rrv_winners])
pltn_positions = ak.flatten(results.electorate.positions[pltn_winners])

In [10]:
factions_df = pd.DataFrame(
    dict(
        rrv_factions=rrv_factions,
        pltn_factions=pltn_factions,
        rrv_pos_x=rrv_positions[:, 0],
        rrv_pos_y=rrv_positions[:, 1],
        pltn_pos_x=pltn_positions[:, 0],
        pltn_pos_y=pltn_positions[:, 1],
    )
)

In [11]:
pn.Column(
    factions_df.hvplot.hist(y="rrv_factions"),
    factions_df.hvplot.hist(y="pltn_factions"),
)

BokehModel(combine_events=True, render_bundle={'docs_json': {'836220e4-25ae-4812-8338-2b0e5c8f5ad9': {'version…

In [12]:
factions_df.hvplot.scatter(x="rrv_pos_x", y="rrv_pos_y", color="rrv_factions")

:Scatter   [rrv_pos_x]   (rrv_pos_y,rrv_factions)

In [13]:
factions_df.hvplot.scatter(x="pltn_pos_x", y="pltn_pos_y", color="pltn_factions")

:Scatter   [pltn_pos_x]   (pltn_pos_y,pltn_factions)